In [2]:
import   torch

In [3]:
# 定义一个字典，将数字标签映射为对应的中文名称
classification_names = {
    0: '上身衣服',  # 数字 0 对应“上身衣服”
    1: '鞋',       # 数字 1 对应“鞋”
    2: '包',       # 数字 2 对应“包”
    3: '下身衣服',  # 数字 3 对应“下身衣服”
    4: '手表'      # 数字 4 对应“手表”
}

In [4]:
import torch 
import re # 导入 PyTorch 库，用于深度学习任务
from PIL import Image  # 导入 PIL 库中的 Image 模块，用于图像处理
import os  # 导入 os 库，用于文件和目录操作
from torch.utils.data import Dataset  # 导入 PyTorch 的 Dataset 类，用于创建自定义数据集
import torchvision.transforms as T  # 导入 torchvision 的 transforms 模块，用于图像预处理
import pandas as pd  # 导入 pandas 库，用于数据处理和分析

In [5]:
def sorted_alphanumeric(data):
    """按字母数字混合顺序对文件名进行排序（例如：img1, img2, ..., img10）"""
    # 定义转换函数：将数字部分转换为整数，非数字部分转换为小写
    convert = lambda text: int(text) if text.isdigit() else text.lower()
    # 生成排序键：用正则分割字符串，分别处理数字和非数字部分
    alphanum_key = lambda key: [convert(c) for c in re.split('([0-9]+)', key)]
    # 按生成的键排序
    return sorted(data, key=alphanum_key)

In [6]:
class ImageDataset(Dataset):
    def __init__(self,main_dir,transform=None):
        """
        初始化方法。
        :param main_dir: 图像文件夹的主路径
        :param transform: 图像预处理操作（如缩放、裁剪、归一化等），默认为 None
        """
        self.main_dir = main_dir
        self.transform = transform
        self.all_imgs = sorted_alphanumeric(os.listdir(self.main_dir))        # 获取主目录下的所有图像文件名
        # 读取标签
        self.classifications = pd.read_csv("../common/fashion-labels.csv")
        self.label_dict = dict(zip(self.classifications['id'],self.classifications['target']))
    def __len__(self):
        return len(self.all_imgs)

    def __getitem__(self, index):
        img_loc = os.path.join(self.main_dir, self.all_imgs[index])  # 获取第 idx 张图像的完整路径
        image = Image.open(img_loc).convert("RGB")  # 打开图像并将其转换为 RGB 格式
        img_flag_id = self.label_dict[index]  # 在分类数据中查找对应的标签

        # 对图像进行预处理（如果定义了 transform）
        if self.transform is not None:
            tensor_image = self.transform(image)  # 应用预处理操作，将图像转换为张量
        else:
            raise RuntimeError("transform参数不能为None，需指定预处理方法")

        return tensor_image, img_flag_id  # 返回图像张量和标签
import torchvision.transforms as T
transforms = T.Compose(
    [
        T.Resize((64,64)),
        T.ToTensor(),
    ]
)
dataset = ImageDataset("../common/dataset",transforms)
print(len(dataset))
print(dataset[0])

24853
(tensor([[[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]],

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]],

        [[1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         ...,
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.],
         [1., 1., 1.,  ..., 1., 1., 1.]]]), 3)


In [7]:
import re

# 检查是否有可用的GPU，如果有则使用GPU，否则使用CPU
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

# 定义数据预处理流程：将图像调整为64x64大小，并将其转换为Tensor
transforms = T.Compose([T.Resize((64, 64)), T.ToTensor()])

# 打印提示信息，表示正在创建数据集
print("------------ 正在创建数据集 ------------")
# 使用ImageDataset类加载数据集，数据集路径为'./image_similarity/dataset/'，并应用定义好的预处理流程
full_dataset = ImageDataset('../common/dataset/', transforms)

------------ 正在创建数据集 ------------


In [8]:
# 计算训练集和验证集的大小，训练集占数据集的75%，验证集占剩余的25%
train_size = int(0.75 * len(full_dataset))
val_size = len(full_dataset) - train_size

# 使用random_split函数将数据集随机划分为训练集和验证集
train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size]
)
# 打印提示信息，表示数据集已创建
print("------------ 数据集创建完成 ------------")

# 打印提示信息，表示正在创建DataLoader
print("------------ 创建数据加载器 ------------")

# 定义批次大小为32
batch = 32
# 创建训练集的DataLoader，批次大小为32，打乱数据顺序，并丢弃最后一个不完整的批次
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch, shuffle=True, drop_last=True
)
for x,y in train_loader:
    print(x.shape,y.shape)
    print()
    break

# 创建验证集的DataLoader，批次大小为32，不打乱数据顺序
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=batch
)

# 创建完整数据集的DataLoader，批次大小为32，不打乱数据顺序
full_loader = torch.utils.data.DataLoader(
    full_dataset, batch_size=batch
)

------------ 数据集创建完成 ------------
------------ 创建数据加载器 ------------
torch.Size([32, 3, 64, 64]) torch.Size([32])



# 定义模型


In [ ]:
import torch.nn as nn

class ImageClassifier(nn.Module):
    def __init__(self,num_classes = 5):
        super().__init__()
        self.num_classes = num_classes
        # 第一层 卷积层
        self.conv1 = nn.Conv2d(in_channels=3,out_channels=8,kernel_size=3, stride=1,padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2,stride=2)

        self.conv2 = nn.Conv2d(in_channels=8,out_channels=16,kernel_size=3,stride=1,padding=1)

        self.fc1 = nn.Linear(in_features=16*16*16, out_features=num_classes)

    def forward(self,x):
        # 第一层：卷积 + Relu激活函数
        x = torch.relu(self.conv1(x))
        # print("经过第一层卷积和ReLU激活函数后的维度：", x.shape)
        # 池化
        x = self.pool(x)
        # print("经过第一层池化后的维度：", x.shape)
        # 第二层：卷积 + ReLu
        x = torch.relu(self.conv2(x))
        # print("经过第二层卷积和ReLU激活函数后的维度：", x.shape)
        # 池化
        x = self.pool(x)
        # print("经过第二层池化后的维度：", x.shape)
        # 第三层，将x平展开，输入到全链接层
        # 将特征图展平为一维向量，保留batch_size维度，其余维度展平
        x = x.reshape(x.shape[0], -1)
        # print("经过一维展平后的维度：", x.shape)
        x = self.fc1(x)
        # print("经过全连接层的维度：", x.shape)
        # 然后经过softmax
        # x = torch.softmax(x, dim=1) # 按照行进行softmax,转换为概率
        # print("经过log_softmax处理后的维度：", x.shape)
        return x

In [10]:
model = ImageClassifier()
print(model)
# 获取训练数据加载器的迭代器
it = iter(train_loader)
# 获取下一个批次的数据
X_batch, y_batch = next(it)
print(X_batch.shape, y_batch.shape)
# 打印模型前向传播输出的形状
print("数据经过模型向前传递后的维度：", model.forward(X_batch).shape)

ImageClassifier(
  (conv1): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=4096, out_features=5, bias=True)
)
torch.Size([32, 3, 64, 64]) torch.Size([32])
数据经过模型向前传递后的维度： torch.Size([32, 5])


In [10]:
criterion = nn.CrossEntropyLoss() # 交叉熵损失
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

epochs = 50
model.to(device)

for epoch in range(1,epochs + 1):
    train_loss = 0.0
    for (x,y) in train_loader:
        x,y = x.to(device),y.to(device)
        optimizer.zero_grad()
        # 前向传播
        ouput = model(x)
        # 计算损失
        loss = criterion(ouput,y)
        # 反向传播计算梯度
        loss.backward()
        # 更新参数
        optimizer.step()
        # 累加当前批次的损失
        train_loss += loss.item() * x.size(0)
        # 计算平均训练损失，总损失除以训练集的批次数量
    train_loss = train_loss / len(train_loader)
    # 打印当前epoch的训练损失
    print('Epoch: {} \tTraining Loss: {:.6f}'.format(epoch, train_loss))

Epoch: 1 	Training Loss: 45.126790
Epoch: 2 	Training Loss: 45.105846
Epoch: 3 	Training Loss: 45.102410
Epoch: 4 	Training Loss: 45.100691
Epoch: 5 	Training Loss: 45.095537
Epoch: 6 	Training Loss: 45.100691
Epoch: 7 	Training Loss: 45.095537
Epoch: 8 	Training Loss: 45.107564
Epoch: 9 	Training Loss: 45.107568
Epoch: 10 	Training Loss: 45.104128
Epoch: 11 	Training Loss: 45.100691
Epoch: 12 	Training Loss: 45.105846
Epoch: 13 	Training Loss: 45.093818
Epoch: 14 	Training Loss: 45.107564
Epoch: 15 	Training Loss: 45.104128
Epoch: 16 	Training Loss: 45.095537
Epoch: 17 	Training Loss: 45.105846
Epoch: 18 	Training Loss: 45.100691
Epoch: 19 	Training Loss: 45.097255
Epoch: 20 	Training Loss: 45.105846
Epoch: 21 	Training Loss: 45.100691
Epoch: 22 	Training Loss: 45.105846
Epoch: 23 	Training Loss: 45.095537
Epoch: 24 	Training Loss: 50.299693
Epoch: 25 	Training Loss: 53.354987
Epoch: 26 	Training Loss: 53.356705
Epoch: 27 	Training Loss: 53.358423
Epoch: 28 	Training Loss: 53.353269
E

In [11]:
import matplotlib.pyplot as plt
# 在Jupyter Notebook中内嵌显示matplotlib图像
%matplotlib inline
# 导入numpy库用于数值计算
import numpy as np

# 获取验证集的一个批次数据
dataiter = iter(val_loader)
images, labels = next(dataiter)

# 将模型移动到CPU上，因为后续操作需要在CPU上执行
model.to('cpu')

# 获取模型对输入图像的输出结果
output = model(images)
# 将输出结果从计算图中分离，并转换为numpy数组
output = output.detach().numpy()
# 打印输出结果的形状
print(output.shape)

# 将输入图像从Tensor转换为numpy数组
images = images.numpy()
# 调整图像的维度顺序，从(C, H, W)变为(H, W, C)，以便matplotlib正确显示
images = np.moveaxis(images, 1, -1)
# 打印调整后的图像形状
print(images.shape)

# 遍历每个图像及其对应的模型输出结果
for img, label in zip(images, output):
    # 显示当前图像
    plt.imshow(img)
    plt.show()
    print("label: ", label)
    print("label.shape: ", label.shape)
    # 打印模型预测的类别名称
    print(classification_names[np.argmax(label)])

(32, 5)
(32, 64, 64, 3)


: 